# 15.3 Fairness and Bias

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 15: Responsible AI & Future Trends

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-15-responsible-ai-and-future-trends/15.03_fairness-and-bias.ipynb)

Companion code for Section 15.3 of the book. Run the cells in order from top to bottom.

**Example 15.3.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

# ── Fairness Audit ───────────────────────────────────────────────
class FairnessAuditor:
    '''Computes fairness metrics across demographic subgroups.'''

    def __init__(self, model, feature_names: list = None):
        self.model         = model
        self.feature_names = feature_names

    def compute_group_metrics(self, X, y, group_labels) -> pd.DataFrame:
        '''Compute performance metrics for each demographic group.'''
        y_pred = self.model.predict(X)
        y_prob = self.model.predict_proba(X)[:,1]

        groups = np.unique(group_labels)
        rows   = []

        for g in groups:
            mask = group_labels == g
            y_g, yp_g, ypr_g = y[mask], y_pred[mask], y_prob[mask]

            tn, fp, fn, tp = confusion_matrix(y_g, yp_g, labels=[0,1]).ravel()

            rows.append({
                'group':         g,
                'n':             mask.sum(),
                'base_rate':     round(y_g.mean(), 4),
                'pos_pred_rate': round(yp_g.mean(), 4),   # Demographic parity
                'tpr':           round(tp/(tp+fn+1e-9), 4),  # Recall = Equalized Odds
                'fpr':           round(fp/(fp+tn+1e-9), 4),  # False alarm
                'precision':     round(tp/(tp+fp+1e-9), 4),  # Predictive parity
                'accuracy':      round((tp+tn)/len(y_g), 4),
                'mean_score':    round(ypr_g.mean(), 4),
            })

        return pd.DataFrame(rows).set_index('group')

    def fairness_gaps(self, metrics_df: pd.DataFrame) -> pd.DataFrame:
        '''Compute max disparity across groups for each metric.'''
        metric_cols = ['pos_pred_rate','tpr','fpr','precision','accuracy']
        gaps = {}
        for col in metric_cols:
            vals = metrics_df[col]
            gaps[col] = {'max': vals.max(), 'min': vals.min(),
                         'disparity': round(vals.max()-vals.min(), 4),
                         'ratio': round(vals.max()/(vals.min()+1e-9), 4)}
        return pd.DataFrame(gaps).T


# ── Simulate biased dataset ───────────────────────────────────────
np.random.seed(42)
n = 5000

# Group A: majority (60%) - historically advantaged
# Group B: minority (40%) - historically disadvantaged
group = np.where(np.random.random(n) < 0.6, 'Group_A', 'Group_B')

# Features - Group B has lower historical scores due to systemic inequality
X = np.random.randn(n, 5)
X[group=='Group_B', 0] -= 0.5   # Lower credit score proxy
X[group=='Group_B', 1] -= 0.3   # Lower income proxy

# True ability unrelated to group after controlling for features
true_ability = X[:,0]*0.4 + X[:,1]*0.3 + X[:,2]*0.2 + np.random.normal(0,0.5,n)
y = (true_ability > np.percentile(true_ability, 50)).astype(int)

# But add historical bias: Group B has lower approval rate in historical labels
historical_bias = np.where(group=='Group_B', -0.4, 0)
y_biased = ((true_ability + historical_bias) > np.percentile(true_ability, 50)).astype(int)

X_tr, X_te, y_tr, y_te, g_tr, g_te = train_test_split(
    X, y_biased, group, test_size=0.3, random_state=42
)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc  = scaler.transform(X_te)

# ── Train model (inherits training data bias) ─────────────────────
model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_tr_sc, y_tr)

# ── Fairness audit ────────────────────────────────────────────────
auditor = FairnessAuditor(model)
metrics = auditor.compute_group_metrics(X_te_sc, y_te, g_te)
gaps    = auditor.fairness_gaps(metrics)

print('=== Fairness Audit Report ===')
print(f'\nGroup Metrics:')
print(metrics.round(4).to_string())
print(f'\nFairness Gaps (disparity across groups):')
print(gaps.round(4).to_string())

# Flag violations
print(f'\nFairness Violations:')
THRESHOLDS = {'pos_pred_rate':0.10,'tpr':0.10,'fpr':0.05,'precision':0.10}
for metric, threshold in THRESHOLDS.items():
    disparity = gaps.loc[metric,'disparity']
    status    = '  VIOLATION' if disparity > threshold else '  PASS'
    print(f'  {metric:18s}: disparity={disparity:.4f}  threshold={threshold}  {status}')

# ── Mitigation: post-processing threshold adjustment ──────────────
print(f'\n=== Fairness Mitigation: Threshold Adjustment ===')
y_prob = model.predict_proba(X_te_sc)[:,1]

# Find group-specific thresholds that equalise TPR
best_thresholds = {}
for g in ['Group_A','Group_B']:
    mask = g_te == g
    y_g, prob_g = y_te[mask], y_prob[mask]
    # Find threshold closest to achieving target TPR (use overall TPR as target)
    target_tpr = (model.predict(X_te_sc) == y_te)[y_te==1].mean()
    best_thresh = 0.5
    best_diff   = 999
    for t in np.arange(0.3, 0.8, 0.01):
        tpr = ((prob_g > t) & (y_g==1)).sum() / (y_g.sum() + 1e-9)
        if abs(tpr - target_tpr) < best_diff:
            best_diff   = abs(tpr - target_tpr)
            best_thresh = t
    best_thresholds[g] = round(best_thresh, 2)

# Apply group-specific thresholds
y_fair = np.zeros(len(y_te), dtype=int)
for g, t in best_thresholds.items():
    mask = g_te == g
    y_fair[mask] = (y_prob[mask] > t).astype(int)

print(f'Thresholds: Group_A={best_thresholds["Group_A"]}  Group_B={best_thresholds["Group_B"]}')

# Recompute fairness after mitigation
from sklearn.base import BaseEstimator
class FairModel(BaseEstimator):
    def __init__(self, base_model, scaler, thresholds):
        self.base_model = base_model
        self.scaler     = scaler
        self.thresholds = thresholds
    def predict(self, X, groups=None):
        prob = self.base_model.predict_proba(X)[:,1]
        if groups is not None:
            pred = np.zeros(len(prob), dtype=int)
            for g, t in self.thresholds.items():
                mask = groups == g
                pred[mask] = (prob[mask] > t).astype(int)
            return pred
        return (prob > 0.5).astype(int)
    def predict_proba(self, X):
        return self.base_model.predict_proba(X)

fair_model    = FairModel(model, scaler, best_thresholds)
y_pred_fair   = fair_model.predict(X_te_sc, groups=g_te)

for g in ['Group_A','Group_B']:
    mask = g_te == g
    acc_before = (model.predict(X_te_sc)[mask] == y_te[mask]).mean()
    acc_after  = (y_pred_fair[mask] == y_te[mask]).mean()
    tpr_before = ((model.predict(X_te_sc)[mask]==1)&(y_te[mask]==1)).sum()/(y_te[mask]==1).sum()
    tpr_after  = ((y_pred_fair[mask]==1)&(y_te[mask]==1)).sum()/(y_te[mask]==1).sum()
    print(f'{g}: accuracy {acc_before:.3f}→{acc_after:.3f}  TPR {tpr_before:.3f}→{tpr_after:.3f}')